# Chapter 20: Adjusting Claims Data for Changing Conditions

Code of Chapter 20 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

## 20.6 Python Walkthrough

In [ ]:
import pandas as pd
from gitriangle import triangle
from gidevelop import average_factors, cumulative_factors, develop
from giadjust import (changing_history, average_case, bs_case_adjust,
                      disposal_ratios, bs_paid_adjust)

policies, claims, tx = changing_history()            # strengthening and faster settlement from 2023
tri = {m: triangle(claims, tx, m, 2021, 2025)
       for m in ("paid", "incurred", "reported_count", "closed_count", "open_count")}

def cdf(t):
    return cumulative_factors(average_factors(t, "volume"), 1.0)

n_ult = develop(tri["reported_count"], cdf(tri["reported_count"]))["ultimate"]
average_case(tri["incurred"], tri["paid"], tri["open_count"]).round(0)
disposal_ratios(tri["closed_count"], n_ult).round(3)

adj_inc = bs_case_adjust(tri["incurred"], tri["paid"], tri["open_count"], 0.04)
adj_paid = bs_paid_adjust(tri["paid"], tri["closed_count"], n_ult)
develop(tri["incurred"], cdf(adj_inc))["ultimate"].round(0)     # B-S incurred
develop(tri["paid"], cdf(adj_paid))["ultimate"].round(0)        # B-S paid